In [ ]:
import subprocess
import tempfile
from itertools import chain
from pathlib import Path

from IPython.display import Markdown, display
from tqdm.notebook import tqdm

import ghapicache
import githelpers
import release_notes

ReleaseMetadata = release_notes.ReleaseMetadata

In [2]:
%load_ext autoreload
%autoreload 1
%aimport githelpers
%aimport ghapicache
%aimport release_notes

In [3]:
local_repo = Path.home() / "Code/celeritas"

# TODO: implement stashed code
# git = githelpers.GitRunner(local_repo)
# release_notes.GIT = git

In [4]:
cached = ghapicache.GhApiCache(repo="celeritas")
ghapi = cached.api
# cached.purge()

In [5]:
ghapi.limit_rem

5000

# Load teams and process members

In [9]:
!open {user_cache.json_path}

In [17]:
# Load teams
TEAMS = {}
for t in tqdm(cached.teams()):
    TEAMS[t["name"]] = release_notes.get_team(cached, t)

team_members = frozenset(chain(*(t.members for t in TEAMS.values())))
org_members = frozenset(m["login"] for m in cached.org_members())
TEAMS["affiliates"] = release_notes.Team(
    description="Associated with Celeritas but not core members",
    members=(org_members - team_members),
)

# Initialize the user cache
user_cache = release_notes.UserCache(
    cached, local_repo / "scripts/release/contributors.jsonl"
)
for login in tqdm(org_members):
    user_cache[login]

  0%|          | 0/6 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

In [18]:
user_cache._metadata

{'amandalund': {'user': 'amandalund',
  'name': 'Amanda Lund',
  'orcid': '0000-0002-8316-0709',
  'institute': 'ANL'},
 'aprokop': {'user': 'aprokop',
  'name': 'Andrey Prokopenko',
  'orcid': '0000-0003-3616-5504',
  'institute': None},
 'dalg24': {'user': 'dalg24',
  'name': 'Damien L-G',
  'orcid': '0000-0003-1952-7219',
  'institute': None},
 'davidsgr': {'user': 'davidsgr',
  'name': 'Greg Davidson',
  'orcid': '0000-0002-3941-8688',
  'institute': 'ORNL'},
 'DoaaDeeb': {'user': 'DoaaDeeb',
  'name': 'Doaa Deeb',
  'orcid': None,
  'institute': 'UC Davis'},
 'drbenmorgan': {'user': 'drbenmorgan',
  'name': 'Ben Morgan',
  'orcid': '0000-0003-3604-0883',
  'institute': 'U Warwick'},
 'elliottbiondo': {'user': 'elliottbiondo',
  'name': 'Elliott Biondo',
  'orcid': '0000-0002-9088-1360',
  'institute': 'ORNL'},
 'erica-snider': {'user': 'erica-snider',
  'name': None,
  'orcid': None,
  'institute': None},
 'esseivaju': {'user': 'esseivaju',
  'name': 'Julien Esseiva',
  'orcid': '

In [19]:
user_cache.flush()

In [71]:
!open {user_cache.json_path}

## List active members

This is to be used for crediting in presentations, etc.

In [9]:
text = []
for team, title in [
    ("code-lead", "Code lead"),
    ("core", "Core members"),
    ("core-advisor", "Core advisors"),
    ("affiliates", "Affiliates"),
]:
    text.append(f"## {title}")
    m = [user_cache[username] for username in TEAMS[team].members]
    for member in sorted(m, key=release_notes.get_last_name):
        text.append("- " + release_notes.format_user(member))
    text.append("")

display(Markdown("\n".join(text)))

## Code lead
- Seth R. Johnson *(ORNL)*

## Core members
- Elliott Biondo *(ORNL)*
- Greg Davidson *(ORNL)*
- Julien Esseiva *(LBNL)*
- Rashika Gupta *(ORNL)*
- Hayden Hollenbeck *(UVA)*
- Seth R. Johnson *(ORNL)*
- Soon Yung Jun *(FNAL)*
- Guilherme Lima *(FNAL)*
- Amanda Lund *(ANL)*
- Ben Morgan *(U Warwick)*
- Sakib Rahman *(BNL)*
- Loren Schwiebert *(Wayne State U)*
- Stefano Tognini *(ORNL)*

## Core advisors
- Philippe Canal *(FNAL)*
- Marcel Demarteau *(ORNL)*
- Tom Evans *(ORNL)*

## Affiliates
- None *(None)*
- Lance Bullerwell *(ORNL)*
- Wouter Deconinck *(U Manitoba)*
- Sam Eriksen *(U Bristol)*
- Sanmay Ganguly *(None)*
- Steven Hamilton *(ORNL)*
- Ilker Parmaksiz *(None)*
- Kevin Pedro *(None)*
- Paul Romano *(ANL)*
- Frederic Suter *(ORNL)*
- Sandro Wenzel *(CERN)*


In [12]:
collaborators = [user_cache[username] for username in user_cache.users]

In [13]:
collaborators[0]._asdict()

{'name': 'Amanda Lund',
 'institute': 'ANL',
 'email': 'alund@anl.gov',
 'orcid': '0000-0002-8316-0709',
 'github': 'amandalund'}

In [14]:
import csv

# Write collaborators to CSV
with open("collaborators.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["last", "first", "orcid", "institution"])

    for collab in sorted(collaborators, key=release_notes.get_last_name):
        # Split name into first and last
        if collab.name:
            name_parts = collab.name.strip().split()
            if len(name_parts) > 1:
                last = name_parts[-1]
                first = " ".join(name_parts[:-1])
            else:
                last = name_parts[0]
                first = ""
        else:
            first = ""
            last = ""

        writer.writerow([last, first, collab.orcid or "", collab.institute or ""])

print(f"Wrote {len(collaborators)} collaborators to collaborators.csv")

Wrote 37 collaborators to collaborators.csv


In [15]:
!open .

## CHEP authors and contributors

In [8]:
# All commits from all time
develop_sha = githelpers.git_rev_parse("upstream/develop")
all_md = ReleaseMetadata(
    release=f"develop-{develop_sha}",
    merge_bases=["v0.0.0"],
    target_branch=develop_sha,
)

all_prs = release_notes.PullRequestRange(all_md)
count_all_contrib = release_notes.ContributionCounter(cached)
for pr in tqdm(all_prs.pull_ids):
    count_all_contrib(pr)
all_contrib_authors = count_all_contrib.sorted().author

Can't match log subject to PR: Format code base (clang-format version 11.0.1)


  0%|          | 0/2061 [00:00<?, ?it/s]

In [9]:
print("Missing ORCIDs:")
print(" ".join(f"@{u}" for u in all_contrib_authors if user_cache[u].orcid is None))

Missing ORCIDs:
@tghaddar @DoaaDeeb


In [10]:
for username in tqdm(set(all_contrib_authors) - set(user_cache.users)):
    print(user_cache[username])

  0%|          | 0/1 [00:00<?, ?it/s]

UserInfo(name=None, institute=None, email=None, orcid=None, github='tghaddar')


In [55]:
cached.flush()
user_cache.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


In [69]:
INST_ADDRESS = {
    "CERN": "CERN, Geneva, Switzerland",
    "ORNL": "Oak Ridge National Laboratory, Oak Ridge, TN, USA",
    "U Warwick": "Department of Physics, University of Warwick, Coventry, United Kingdom",
    "FNAL": "Fermi National Accelerator Laboratory, Batavia, IL, USA",
    "ANL": "Argonne National Laboratory, Lemont, IL, USA",
    "LBNL": "Lawrence Berkeley National Laboratory, Berkeley, CA, USA",
    "UVA": "University of Virginia, Charlottesville, VA, USA",
    "Wayne State U": "Wayne State University, Detroit, MI, USA",
    "BNL": "Brookhaven National Laboratory, Upton, NY, USA",
}


def ordered_uniquify(x: list):
    result: list = []
    seen = set()

    for v in x:
        if v in seen:
            continue
        result.append(v)
        seen.add(v)

    return result


def print_chep_authors(contributors, institutes):
    """Print CHEP author blocks given contributors and ordered institute list."""
    institute_map = {inst: i for i, inst in enumerate(institutes, start=1)}
    author_blocks = []

    for c in contributors:
        parts = c.name.split()
        if len(parts) > 1:
            first = " ".join(parts[:-1])
            last = parts[-1]
        else:
            first = ""
            last = parts[0]

        inst_index = institute_map[c.institute]
        inst_suffix = f"\\inst{{{inst_index}}}"
        orcid_suffix = f"\n  \\orcidlink{{{c.orcid}}}%" if c.orcid else ""

        author_blocks.append(
            f"  \\firstname{{{first}}} \\lastname{{{last}}}{inst_suffix}%{orcid_suffix}"
        )

    if not author_blocks:
        print("  No authors found")
        return

    print("\n  %%\n  \\and\n".join(author_blocks))


def print_chep_institutes(institutes):
    """Print institute addresses joined with \and."""
    addresses = []
    for institute in institutes:
        addresses.append(INST_ADDRESS[institute])

    if not addresses:
        print("  No institutes found")
        return

    print("\n  \\and\n".join(f"  {addr}" for addr in addresses))

In [70]:
chep: list[release_notes.UserInfo] = []
for team in ["code-lead", "core", "core-advisor"]:
    users = [user_cache[username] for username in TEAMS[team].members]
    users.sort(key=lambda u: u.name.split()[-1])
    for u in users:
        if u.institute is None:
            print(f"missing insitute: {u}")
            continue
        chep.append(u)

chep

print({u.institute for u in chep} - set(INST_ADDRESS))

set()


In [ ]:
institutes = ordered_uniquify(["CERN", "ORNL"] + [u.institute for u in chep])
print_chep_authors(chep, institutes)
print_chep_institutes(institutes)

  \firstname{Seth R.} \lastname{Johnson}\inst{2}%
  \orcidlink{0000-0003-1504-8966}%
  %%
  \and
  \firstname{Elliott} \lastname{Biondo}\inst{2}%
  \orcidlink{0000-0002-9088-1360}%
  %%
  \and
  \firstname{Greg} \lastname{Davidson}\inst{2}%
  \orcidlink{0000-0002-3941-8688}%
  %%
  \and
  \firstname{Julien} \lastname{Esseiva}\inst{3}%
  \orcidlink{0009-0002-1119-4851}%
  %%
  \and
  \firstname{Rashika} \lastname{Gupta}\inst{2}%
  \orcidlink{0009-0003-0658-9549}%
  %%
  \and
  \firstname{Hayden} \lastname{Hollenbeck}\inst{4}%
  \orcidlink{0009-0000-6359-3294}%
  %%
  \and
  \firstname{Seth R.} \lastname{Johnson}\inst{2}%
  \orcidlink{0000-0003-1504-8966}%
  %%
  \and
  \firstname{Soon Yung} \lastname{Jun}\inst{5}%
  \orcidlink{0000-0003-3370-6109}%
  %%
  \and
  \firstname{Guilherme} \lastname{Lima}\inst{5}%
  \orcidlink{0000-0003-4585-0546}%
  %%
  \and
  \firstname{Amanda} \lastname{Lund}\inst{6}%
  \orcidlink{0000-0002-8316-0709}%
  %%
  \and
  \firstname{Ben} \lastname{Morgan}\inst{

# Release process

- Merge base should be all commits *already* released (skip documenting)
- Target branch is the one where the release candidate is
- Previous major branch allows all "v.x" contributors to be credited

In [20]:
# Major release
major_md = release_notes.ReleaseMetadata(
    release="0.7.0", merge_bases=["v0.6.0", "v0.6.3"], target_branch="upstream/develop"
)

# Minor release
minor_md = release_notes.ReleaseMetadata(
    release="0.6.3", merge_bases=["v0.6.2"], target_branch="backports/v0.6"
)

## Release note generation

In [21]:
if 0:
    # Backport release:
    release_md = minor_md
    note_body = """
Version {release} is a minor bug fix update.

- Support CUDA 13
- Allow initialization of Geant4 integration before run manager
"""
else:
    # Major release
    release_md = major_md
    note_body = """
Version {release} is a major update to Celeritas featuring many new capabilities.

-

A few minor features are noteworthy:

- 

Notable bug fixes include:

- 

Some interfaces have been removed:

- 

Known deficiencies:

- 
"""

In [29]:
pr_parser = release_notes.PullRequestLogParser(Path("data/github-sha-map.json"))
prs = release_notes.PullRequestRange(release_md, pr_parser)
sorted_pulls = release_notes.SortedPulls(cached)
count_contrib = release_notes.ContributionCounter(cached)
for pr_id in tqdm(prs.pull_ids):
    try:
        count_contrib(pr_id)
        sorted_pulls.add(pr_id)
    except Exception as e:
        print(f"Error adding PR #{pr_id}: {e}")
        del cached.cache["pull"][cached.subkey(pr_id)]

reviewers = count_contrib.sorted().reviewer
for login in tqdm(reviewers):
    user_cache[login]

Ignore b427d6915c82e20f2c1312b9d647ece6fce3d6ae: presets on backport branch
Ignore cc6f24d1716c9e2f652f9db3e732cf82f056c76e: backported component from #1895


  0%|          | 0/599 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

In [23]:
cached.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


In [25]:
def make_notes(cls, release_md, note_body):
    prev = release_md.merge_bases[0]
    other = release_md.merge_bases[1:]
    if not other:
        change_str = f"Changes since {prev} follow."
    else:
        # NOTE: this is redundant if prev is a merge base of other
        assert len(other) == 1
        change_str = (
            f"Changes since {prev}, excluding those released in {other[0]}, follow."
        )

    notes = cls(release_md, note_body)
    notes.paragraph(change_str)
    notes.sorted_pulls(sorted_pulls)
    notes.reviewers(reviewers, user_cache)
    notes.changelog_line("celeritas-project", "celeritas")

    return notes

In [26]:
rst_notes = make_notes(release_notes.RstNotes, release_md, note_body)

with githelpers.open_pbcopy() as pb:
    rst_notes.write(pb)

print("Release notes copied to clipboard")
subprocess.run(["open", str(githelpers.REPO / "doc/release-history")])

KeyError: 'Params,State'

## Create github release

For backport releases, make sure the release history above is *first* merged into upstream **and cherry-picked into the backport branch**.

First, tag the release locally and update the target to the tagged commit.

In [ ]:
release_notes.create_local_tag(release_md)
subject = githelpers.git("log", "-1", r"--format=%s", release_md.target_branch)[0]
if not subject.startswith("Release v" + release_md.release):
    print(
        f"WARNING: expected last commit subject to include release, but subject line is '{subject}'"
    )

Updating release target from backports/v0.6 to ddf6c690e37d53b56d001e6bf0c26cb1b14963fa


Then create a gzipped in-memory archive of the repository. Decompress and run `cmake -P cmake/CgvFindVersion.cmake` to check the version is exported correctly.

In [72]:
tgz_data = githelpers.git_archive_tgz(release_md.target_branch)
tgz_name = f"{cached.repo}-{release_md.release}.tar.gz"

with tempfile.TemporaryDirectory(delete=False) as tmpdir:
    with open(Path(tmpdir) / tgz_name, "wb") as f:
        f.write(tgz_data)
    subprocess.run(["open", tmpdir])

In [74]:
subprocess.run(
    ["cmake", "-P", "cmake/CgvFindVersion.cmake"],
    cwd=Path(tmpdir) / f"{cached.repo}-{release_md.release}",
)

VERSION="0.7.0"
VERSION_STRING="0.7.0-rc.1"


CompletedProcess(args=['cmake', '-P', 'cmake/CgvFindVersion.cmake'], returncode=0)

Now generate the PDF documentation and check it.

In [ ]:
build_dir = local_repo / "build"


def run_build(*args):
    return subprocess.run(args, cwd=build_dir, check=True)


run_build("cmake", "-UCeleritas_GIT_DESCRIBE", ".")
run_build("ninja", "doc_pdf")
pdf_doc = build_dir / "doc" / "latex" / "Celeritas.pdf"
assert pdf_doc.is_file()
!open {pdf_doc}

-- Configuring done (0.5s)
-- Generating done (0.2s)
-- Build files have been written to: /Users/seth/Code/celeritas-temp/build
[1/6] Performing update step for 'ornltm'
-- Already at requested ref: 6379032d2f1e5e8bc1589d087465371fd8b432c1
[2/6] No patch step for 'ornltm'
[3/6] No configure step for 'ornltm'
[4/6] No build step for 'ornltm'
[5/6] No install step for 'ornltm'
[6/6] Completed 'ornltm'


Create the github release draft and upload the tarball and PDF documentation

In [ ]:
ghapi = cached.api

markdown_notes = make_notes(release_notes.MarkdownNotes, release_md, note_body)

assert release_md.release is not None
gh_release = release_notes.find_release(ghapi, release_md.release)
if not gh_release:
    gh_release = release_notes.create_release(ghapi, release_md, str(markdown_notes))
    tgz_asset = release_notes.upload_asset(cached, gh_release, tgz_data, tgz_name)
    doc_asset = release_notes.upload_asset(
        cached, gh_release, pdf_doc, f"celeritas-user-manual-{release_md.release}.pdf"
    )

No release found for version: 0.6.3
Draft release 0.6.3 created: https://github.com/celeritas-project/celeritas/releases/tag/untagged-804e7f799a631819ea29
Uploaded artifact: https://github.com/celeritas-project/celeritas/releases/download/untagged-804e7f799a631819ea29/celeritas-0.6.3.tar.gz
Uploaded artifact: https://github.com/celeritas-project/celeritas/releases/download/untagged-804e7f799a631819ea29/celeritas-user-manual-0.6.3.pdf


In [ ]:
sha256_hash = release_notes.prepare_spack_update("g4vg", release_md, tgz_data)

Spack version copied to clipboard!


0

In [ ]:
display(
    Markdown(f"""
## Time to publish!

1. Visit [the github release page]({gh_release["html_url"]})
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is {sha256_hash}
4. Double check the name and release notes
5. Edit and publish!
""")
)


## Time to publish

1. Visit [the github release page](https://github.com/celeritas-project/celeritas/releases/tag/untagged-804e7f799a631819ea29)
2. Verify the tgz and pdf are both created
3. Ensure the tgz SHA256 sum is c87910acd0bb9dc85c53e2e4d9958f36d3bb1e5975f0e08fe7274b332a2ccf27
4. Double check the name and release notes
5. Hit publish!


## Print authors since the last major release for presentations

In [ ]:
latest_major_md = ReleaseMetadata(
    release=f"recent-{release_md.target_branch}",
    merge_bases=["v0.7.0-dev"],
    target_branch="upstream/develop",
    # merge_bases=['v0.6.0-dev'],
    # target_branch='backports/v0.6',
)

latest_prs = release_notes.PullRequestRange(latest_major_md)
count_contrib = release_notes.ContributionCounter(cached)
for pr in tqdm(latest_prs.pull_ids):
    count_contrib(pr)
authors = count_contrib.sorted().author

text = ["# Current contributors"]
current_contrib = set()
for username in authors:
    m = user_cache[username]
    text.append(f"- {m.name} (@{m.github})")
    current_contrib.add(m.github)
text.extend(["", "# Past contributors"])
for username in all_contrib_authors:
    if username not in current_contrib:
        m = user_cache[username]
        text.append(f"- {m.name} (@{username})")
# display(Markdown("\n".join(text)))
print("\n".join(text))

  0%|          | 0/494 [00:00<?, ?it/s]

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Elliott Biondo (@elliottbiondo)
- Julien Esseiva (@esseivaju)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Owen Strong (@osanstrong)
- Ben Morgan (@drbenmorgan)
- Sakib Rahman (@rahmans1)
- Guilherme Lima (@mrguilima)
- Lance Bullerwell (@lebuller)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)
- Peter Heywood (@ptheywood)


In [19]:
display(Markdown("\n".join(text)))

# Current contributors
- Seth R. Johnson (@sethrj)
- Amanda Lund (@amandalund)
- Elliott Biondo (@elliottbiondo)
- Julien Esseiva (@esseivaju)
- Hayden Hollenbeck (@hhollenb)
- Rashika Gupta (@Rashika-Gupta)
- Philippe Canal (@pcanal)
- Stefano Tognini (@stognini)
- Loren Schwiebert (@LSchwiebert)
- Soon Yung Jun (@whokion)
- Greg Davidson (@davidsgr)
- Owen Strong (@osanstrong)
- Ben Morgan (@drbenmorgan)
- Sakib Rahman (@rahmans1)
- Guilherme Lima (@mrguilima)
- Lance Bullerwell (@lebuller)
- Stephen Nicholas Swatman (@stephenswat)

# Past contributors
- Paul Romano (@paulromano)
- Vincent R. Pascuzzi (@vrpascuzzi)
- Tom Evans (@tmdelellis)
- Vidor Heli Lujan Montiel (@VHLM2001)
- Damien L-G (@dalg24)
- Doaa Deeb (@DoaaDeeb)
- Andrey Prokopenko (@aprokop)
- Shane Hart (@hartsw)
- Peter Heywood (@ptheywood)

In [18]:
cached.flush()

Saved cache to: data/ghapicache-celeritas-project-celeritas.json


# Push to zenodo

In [17]:
import zenodoapi

%aimport zenodoapi

In [18]:
# Load the Zenodo token
token_path = Path.home() / ".config/zenodo-token"
with open(token_path) as f:
    zenodo_token = f.read().strip()

zenodo = zenodoapi.Zenodo(zenodo_token)
zenodo.api_url = "https://zenodo.org/api/"

In [ ]:
def load_contributions(release_md):
    prs = release_notes.PullRequestRange(release_md)
    count_contrib = release_notes.ContributionCounter(cached)
    for pr in tqdm(prs.pull_ids):
        count_contrib(pr)

    # Create author list
    return count_contrib.sorted()


def download_tarball(gh_release):
    asset = gh_release["assets"][0]
    name = asset["name"]
    content = cached.download_file(asset["url"], ext=Path(name).suffix)
    assert content is not None
    return (content, name)


## Minor release

This assumes the major release has already been published upstream into Zenodo.

In [ ]:
assert not release_md.is_major()
major, minor, patch = release_md.as_version()
release_md = release_notes.ReleaseMetadata.from_comprehensive_version(minor, patch)
major_dep = zenodo.find_deposition(f"Celeritas {major}.{minor}")

TypeError: cannot unpack non-iterable NoneType object

In [29]:
zenodo.get_deposition(major_dep.get_latest_version().id)

ZenodoDeposition(id=17193315, title="Celeritas 0.6")

In [30]:
gh_release = release_notes.find_release(ghapi, release_md.release)

In [ ]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)

In [ ]:
# Get the Zenodo metadata
from requests import HTTPError


old_md = major_dep.data["metadata"]
new_md = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
# Only update editors, not team members
old_contrib = new_md["contributors"]
new_contrib = [u for u in old_contrib if u["type"] == "Editor"]
old_contrib = [u for u in old_contrib if u["type"] != "Editor"]
new_md["contributors"] = new_contrib + old_contrib
# Don't change the title
new_md["title"] = major_dep.data["metadata"]["title"]
# Add the body
new_md["description"] = "\n\n".join([
    old_md["description"],
    f"<h1>Version {release_md.release}</h1>",
    new_md["description"],
])
try:
    new_vers = major_dep.create_new_version()
except HTTPError as e:
    # Maybe the version already exists
    if e.response.status_code == 400:
        print("Draft may already exist:", e)
        new_vers = major_dep.get_latest_draft()
    else:
        raise
try:
    new_vers.update(new_md)
except HTTPError as e:
    # Something else went wrong? Try to upload the tarball
    print("Draft may already exist:", e)

# Upload the release
(content, name) = download_tarball(gh_release)
new_vers.upload(content, name)
# The old tarball may still be there (this is buggy) so delete it
new_vers.refresh()
for file in new_vers.get_files():
    if file.filename != name:
        try:
            file.delete()
        except Exception as e:
            print(f"Failed to delete {file.filename}: {e}")
new_vers.refresh()

Can't match log subject to PR: Update cmake presets
Can't match log subject to PR: Write setup Geant4 log messages to world logger, not self logger


  0%|          | 0/369 [00:00<?, ?it/s]

Created new version 17684516: https://zenodo.org/deposit/17684516
Updated deposition at https://zenodo.org/deposit/17684516 : Celeritas 0.6
Uploaded celeritas-0.6.3.tar.gz: version 3056d2d2-889c-4044-8db5-631e0f197235
Failed to delete celeritas-0.6.2.tar.gz: Expecting value: line 1 column 1 (char 0)


ZenodoDeposition(id=17684516, title="Celeritas 0.6")

## Major release

In [ ]:
release_md = release_notes.ReleaseMetadata(
    release="0.7.0-rc.1",
    # merge_bases=["v0.6.0", "v0.6.3"],
    merge_bases=["v0.7.0-dev^"],
    target_branch="upstream/develop",
)

In [36]:
release_md.as_version()

(0, 7, 0)

In [ ]:
assert release_md is not None and release_md.is_major()
major, minor, patch = release_md.as_version()
# release_md = release_notes.ReleaseMetadata.from_comprehensive_version(major, minor)
title = f"Celeritas {major}.{minor}"

In [41]:
release_md

ReleaseMetadata(release='0.7.0-rc.1', merge_bases=['v0.7.0-dev^'], target_branch='upstream/develop')

In [44]:
gh_release = release_notes.find_release(ghapi, release_md.release)

No release found for version: 0.7.0-rc.1


In [47]:
load_contributions(release_md)

  0%|          | 0/550 [00:00<?, ?it/s]

Contributions(author={'sethrj': 288, 'amandalund': 75, 'esseivaju': 46, 'elliottbiondo': 37, 'hhollenb': 27, 'pcanal': 14, 'Rashika-Gupta': 13, 'stognini': 11, 'LSchwiebert': 8, 'whokion': 6, 'davidsgr': 6, 'rahmans1': 5, 'drbenmorgan': 4, 'osanstrong': 4, 'mrguilima': 2, 'tghaddar': 2, 'lebuller': 1, 'stephenswat': 1}, reviewer={'sethrj': 240, 'amandalund': 108, 'pcanal': 57, 'elliottbiondo': 56, 'esseivaju': 45, 'hhollenb': 22, 'stognini': 22, 'mrguilima': 15, 'Rashika-Gupta': 12, 'whokion': 5, 'rahmans1': 3, 'drbenmorgan': 2, 'davidsgr': 2, 'LSchwiebert': 2})

In [48]:
sorted(_.author.items(), key=lambda x: x[1])

[('lebuller', 1),
 ('stephenswat', 1),
 ('mrguilima', 2),
 ('tghaddar', 2),
 ('drbenmorgan', 4),
 ('osanstrong', 4),
 ('rahmans1', 5),
 ('whokion', 6),
 ('davidsgr', 6),
 ('LSchwiebert', 8),
 ('stognini', 11),
 ('Rashika-Gupta', 13),
 ('pcanal', 14),
 ('hhollenb', 27),
 ('elliottbiondo', 37),
 ('esseivaju', 46),
 ('amandalund', 75),
 ('sethrj', 288)]

In [54]:
make_zenodo_md = release_notes.ZenodoMetadataBuilder(user_cache=user_cache, teams=TEAMS)
zmd = make_zenodo_md(load_contributions(release_md), release_md, gh_release)

  0%|          | 0/550 [00:00<?, ?it/s]

ValueError: Missing properties from user tghaddar: UserInfo(name=None, institute=None, email=None, orcid=None, github='tghaddar')

In [50]:
zmd

{'upload_type': 'software',
 'creators': [{'name': 'Seth R. Johnson',
   'affiliation': 'ORNL',
   'orcid': '0000-0003-1504-8966'},
  {'name': 'Amanda Lund',
   'affiliation': 'ANL',
   'orcid': '0000-0002-8316-0709'},
  {'name': 'Julien Esseiva',
   'affiliation': 'LBNL',
   'orcid': '0009-0002-1119-4851'},
  {'name': 'Elliott Biondo',
   'affiliation': 'ORNL',
   'orcid': '0000-0002-9088-1360'},
  {'name': 'Hayden Hollenbeck',
   'affiliation': 'UVA',
   'orcid': '0009-0000-6359-3294'},
  {'name': 'Philippe Canal',
   'affiliation': 'FNAL',
   'orcid': '0000-0002-7748-7887'},
  {'name': 'Rashika Gupta',
   'affiliation': 'ORNL',
   'orcid': '0009-0003-0658-9549'},
  {'name': 'Stefano Tognini',
   'affiliation': 'ORNL',
   'orcid': '0000-0001-9741-6608'},
  {'name': 'Loren Schwiebert',
   'affiliation': 'Wayne State U',
   'orcid': '0000-0002-7167-7436'},
  {'name': 'Soon Yung Jun',
   'affiliation': 'FNAL',
   'orcid': '0000-0003-3370-6109'},
  {'name': 'Greg Davidson',
   'affiliati

In [45]:
zmd = make_zenodo_md(load_contributions(release_md), release_md, gh_release)
zmd["title"] = title

# Create the Zenodo deposition
deposition = zenodo.create_deposition(zmd)

  0%|          | 0/550 [00:00<?, ?it/s]

Created deposition 23104914 at https://zenodo.org/deposit/23104914 : Celeritas 0.7


In [46]:
zmd

{'upload_type': 'software',
 'creators': [{'name': 'Seth R. Johnson',
   'affiliation': 'ORNL',
   'orcid': '0000-0003-1504-8966'},
  {'name': 'Amanda Lund',
   'affiliation': 'ANL',
   'orcid': '0000-0002-8316-0709'},
  {'name': 'Julien Esseiva',
   'affiliation': 'LBNL',
   'orcid': '0009-0002-1119-4851'},
  {'name': 'Elliott Biondo',
   'affiliation': 'ORNL',
   'orcid': '0000-0002-9088-1360'},
  {'name': 'Hayden Hollenbeck',
   'affiliation': 'UVA',
   'orcid': '0009-0000-6359-3294'},
  {'name': 'Philippe Canal',
   'affiliation': 'FNAL',
   'orcid': '0000-0002-7748-7887'},
  {'name': 'Rashika Gupta',
   'affiliation': 'ORNL',
   'orcid': '0009-0003-0658-9549'},
  {'name': 'Stefano Tognini',
   'affiliation': 'ORNL',
   'orcid': '0000-0001-9741-6608'},
  {'name': 'Loren Schwiebert',
   'affiliation': 'Wayne State U',
   'orcid': '0000-0002-7167-7436'},
  {'name': 'Soon Yung Jun',
   'affiliation': 'FNAL',
   'orcid': '0000-0003-3370-6109'},
  {'name': 'Greg Davidson',
   'affiliati

In [ ]:
# Upload the release tarball
deposition.upload(artifact_tgz, f"celeritas-{release_md.release}.tar.gz")

print(deposition.html)
print("""
IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses
""")

Uploaded celeritas-0.6.0.tar.gz: version bbd63016-30a1-4be9-9a74-329ca975bed9
https://zenodo.org/deposit/15281110

IMPORTANT Checklist:
- Add community (CANNOT be done later)
- Update contributors based on release date
- Update licenses

